In [104]:
import requests
import pandas as pd
import time

BASE_URL = "https://fapi.binance.com/fapi/v1/klines"

def get_klines(symbol, interval, start_ts, end_ts=None):
    all_data = []
    while True:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_ts,
            "limit": 1500
        }
        if end_ts:
            params["endTime"] = end_ts

        r = requests.get(BASE_URL, params=params)
        data = r.json()

        if not data:
            break

        all_data.extend(data)
        start_ts = data[-1][0] + 1

        if len(data) < 1500:
            break

        time.sleep(0.05)  # 防止被限流

    df = pd.DataFrame(all_data, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades",
        "taker_base_vol", "taker_quote_vol", "ignore"
    ])

    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
    df = df[["timestamp", "open", "high", "low", "close", "volume"]]
    df = df.astype({"open": float, "high": float,
                     "low": float, "close": float,
                     "volume": float})
    return df

# 示例：拉 7 天数据
start_time = int(pd.Timestamp("2025-06-01").timestamp() * 1000)
end_time = int(pd.Timestamp("2026-01-11").timestamp() * 1000)
df_1m = get_klines("RIVERUSDT", "1m", start_time, end_time)
df_1m.to_csv("RIVERUSDT_data_1m.csv", index=False)


ConnectTimeout: HTTPSConnectionPool(host='fapi.binance.com', port=443): Max retries exceeded with url: /fapi/v1/klines?symbol=RIVERUSDT&interval=1m&startTime=1748736000000&limit=1500&endTime=1768089600000 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x28c5ba710>, 'Connection to fapi.binance.com timed out. (connect timeout=None)'))

In [ ]:
"""
策略：移动止损 双向（简易版）
1. 15m 趋势判断（多头：HH + HL + EMA 多头排列；空头：LL + LH + EMA 空头排列）
2. 1m 入场条件：
   多头：价格回落到 EMA20 且成交量放大
   空头：价格反弹到 EMA20 且成交量放大
3. 移动止损：
   多头：用最近 HL 的 low 计算止损位
   空头：用最近 HH 的 high 计算止损位
"""

import pandas as pd
import numpy as np

# ======================
# 参数区
# ======================
FEE_RATE = 0.0005        # 0.05% taker or maker fee
LEVERAGE = 20
ATR_MULTIPLIER = 10      # ATR 乘数，用于止损计算
RR = 1.5                # risk-reward (未在移动止损版使用，但保留)
HL_LOOKBACK = 5      # 用多少根 1m K 线判断 HL (或 HH) ，默认15
HL_BUFFER = 1.01     # HL + 1% (用于多头移动止损)
HH_BUFFER = 0.99     # HH -1% (用于空头移动止损)

# ======================
# 技术指标
# ======================
def EMA(series, period):
    return series.ewm(span=period, adjust=False).mean()

def ATR(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ======================
# 加载 1m 数据
# ======================
df_1m = pd.read_csv("BTCUSDTm9_m1_data_1m.csv", parse_dates=['timestamp'])
df_1m.set_index('timestamp', inplace=True)

# ======================
# 构建 15m 数据
# ======================
df_15m = df_1m.resample('15T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})

# ======================
# 15m 因子
# ======================
df_15m['EMA20'] = EMA(df_15m['close'], 20)
df_15m['EMA50'] = EMA(df_15m['close'], 50)
df_15m['EMA200'] = EMA(df_15m['close'], 200)

# 结构因子
N = 20
df_15m['HH'] = df_15m['high'] > df_15m['high'].rolling(N).max().shift()
df_15m['HL'] = df_15m['low']  > df_15m['low'].rolling(N).min().shift()
# 空头对应的结构因子（近似取反）
df_15m['LL'] = df_15m['low'] < df_15m['low'].rolling(N).min().shift()
df_15m['LH'] = df_15m['high'] < df_15m['high'].rolling(N).max().shift()

df_15m['ALLOW_LONG'] = (
    df_15m['HH'] &
    df_15m['HL'] &
    (df_15m['close'] > df_15m['EMA20']) &
    (df_15m['EMA20'] > df_15m['EMA50']) &
    (df_15m['EMA50'] > df_15m['EMA200'])
)

df_15m['ALLOW_SHORT'] = (
    df_15m['LL'] &
    df_15m['LH'] &
    (df_15m['close'] < df_15m['EMA20']) &
    (df_15m['EMA20'] < df_15m['EMA50']) &
    (df_15m['EMA50'] < df_15m['EMA200'])
)

# 对齐到 1m
df_1m['ALLOW_LONG'] = df_15m['ALLOW_LONG'].reindex(df_1m.index, method='ffill')
df_1m['ALLOW_SHORT'] = df_15m['ALLOW_SHORT'].reindex(df_1m.index, method='ffill')

# ======================
# 1m 因子
# ======================
df_1m['EMA20'] = EMA(df_1m['close'], 20)
df_1m['ATR'] = ATR(df_1m)

df_1m['VOL_SPIKE'] = (
    df_1m['volume'] >
    1.2 * df_1m['volume'].rolling(20).mean()
)

# ======================
# 回测主循环（移动止损 双向）
# ======================
trades = []
position = None

for t in range(20, len(df_1m)):
    row = df_1m.iloc[t]

    # ===== 开仓 =====
    if position is None:
        # 多头入场：15m 趋势允许，1m 回落到 EMA20 且成交量放大
        if row['ALLOW_LONG'] and row['low'] <= row['EMA20'] and row['VOL_SPIKE']:
            entry = row['close']
            atr = row['ATR']
            stop = entry - ATR_MULTIPLIER * atr
            position = {
                'side': 'long',
                'entry_time': df_1m.index[t],
                'entry_price': entry,
                'stop': stop
            }
        # 空头入场：15m 趋势允许，1m 反弹到 EMA20 且成交量放大
        elif row['ALLOW_SHORT'] and row['high'] >= row['EMA20'] and row['VOL_SPIKE']:
            entry = row['close']
            atr = row['ATR']
            stop = entry + ATR_MULTIPLIER * atr
            position = {
                'side': 'short',
                'entry_time': df_1m.index[t],
                'entry_price': entry,
                'stop': stop
            }

    # ===== 持仓管理 =====
    else:
        low = row['low']
        high = row['high']

        if position['side'] == 'long':
            # 更新多头移动止损（用最近若干根 low）
            recent_lows = df_1m['low'].iloc[t-HL_LOOKBACK:t]
            if len(recent_lows) > 0 and low > recent_lows.min():
                hl_stop = recent_lows.min() * HL_BUFFER
                position['stop'] = max(position['stop'], hl_stop)
            # 检查止损
            if low <= position['stop']:
                exit_price = position['stop']
                gross_ret = (exit_price - position['entry_price']) / position['entry_price']
                net_ret = LEVERAGE * gross_ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'long',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net_ret,
                    'reason': 'trailing_stop'
                })
                position = None

        else:  # short
            # 更新空头移动止损（用最近若干根 high）
            recent_highs = df_1m['high'].iloc[t-HL_LOOKBACK:t]
            if len(recent_highs) > 0 and high < recent_highs.max():
                hh_stop = recent_highs.max() * HH_BUFFER
                # 空头的 stop 在价格上方，向下移动则用 min
                position['stop'] = min(position['stop'], hh_stop)
            # 检查止损（空头触及上方止损）
            if high >= position['stop']:
                exit_price = position['stop']
                # 空头收益 = entry - exit
                gross_ret = (position['entry_price'] - exit_price) / position['entry_price']
                net_ret = LEVERAGE * gross_ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'short',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net_ret,
                    'reason': 'trailing_stop'
                })
                position = None

# ======================
# 结果统计
# ======================
df_trades = pd.DataFrame(trades)

print("交易次数:", len(df_trades))
print("胜率:", (df_trades['return'] > 0).mean())
print("累计收益:", df_trades['return'].sum())
print("最大回撤:", (df_trades['return'].cumsum().cummax() -
                   df_trades['return'].cumsum()).max())


/var/folders/h_/2rpkgm1n1y5gpfss71sfwmgr0000gn/T/ipykernel_36528/3252726101.py:48: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_15m = df_1m.resample('15T').agg({


交易次数: 465
胜率: 0.9956989247311828
累计收益: 72.97006881335555
最大回撤: 0.025583392224529433


In [38]:
"""
策略：移动止损 双向（根据市场状态过滤）
1. 根据市场状态过滤（STRONG_UP 只做多，STRONG_DOWN 只做空）
2. 15m 趋势判断（多头：HH + HL + EMA 多头排列；空头：LL + LH + EMA 空头排列）
3. 1m 入场条件：
   多头：价格回落到 EMA20 且成交量放大
   空头：价格反弹到 EMA20 且成交量放大
4. 移动止损：
   多头：用最近 HL 的 low 计算止损位
   空头：用最近 HH 的 high 计算止损位
"""
import pandas as pd
import numpy as np

# ======================
# 参数区
# ======================
FEE_RATE = 0.0005       # 0.05% taker
LEVERAGE = 20

ATR_MULTIPLIER = 10      # ATR 乘数，用于止损计算
HL_LOOKBACK = 20     # 用多少根 1m K 线判断 HL (或 HH)，默认15
HL_BUFFER = 0.995    # 用于多头移动止损
HH_BUFFER = 1.005  # 用于空头移动止损

# ======================
# 技术指标
# ======================
def EMA(series, period):
    return series.ewm(span=period, adjust=False).mean()

def ATR(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ======================
# 加载 1m 数据
# ======================
df_1m = pd.read_csv("ETHUSDT_data_1m.csv", parse_dates=['timestamp'])
df_1m.set_index('timestamp', inplace=True)

# ======================#
# 构建 15m 数据
# ======================
df_15m = df_1m.resample('15T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}).dropna()

# ======================
# 15m 技术因子
# ======================
df_15m['EMA20'] = EMA(df_15m['close'], 20)
df_15m['EMA50'] = EMA(df_15m['close'], 50)
df_15m['EMA200'] = EMA(df_15m['close'], 200)

# 结构因子
N = 20
df_15m['HH'] = df_15m['high'] > df_15m['high'].rolling(N).max().shift()
df_15m['HL'] = df_15m['low']  > df_15m['low'].rolling(N).min().shift()
df_15m['LL'] = df_15m['low']  < df_15m['low'].rolling(N).min().shift()
df_15m['LH'] = df_15m['high'] < df_15m['high'].rolling(N).max().shift()

df_15m['ALLOW_LONG'] = (
    df_15m['HH'] &
    df_15m['HL'] &
    (df_15m['EMA20'] > df_15m['EMA50']) &
    (df_15m['EMA50'] > df_15m['EMA200'])
)

df_15m['ALLOW_SHORT'] = (
    df_15m['LL'] &
    df_15m['LH'] &
    (df_15m['EMA20'] < df_15m['EMA50']) &
    (df_15m['EMA50'] < df_15m['EMA200'])
)

# ======================
# REGIME（市场状态）
# ======================
df_15m['EMA20_slope'] = (
    df_15m['EMA20'] - df_15m['EMA20'].shift(5)
) / df_15m['close']

def market_regime(row):
    if (
        row['close'] > row['EMA200'] and
        row['EMA20'] > row['EMA50'] > row['EMA200'] and
        row['EMA20_slope'] > 0.0001
    ):
        return 'STRONG_UP'

    elif (
        row['close'] < row['EMA200'] and
        row['EMA20'] < row['EMA50'] < row['EMA200'] and
        row['EMA20_slope'] < -0.0001
    ):
        return 'STRONG_DOWN'

    elif abs(row['EMA20_slope']) < 0.00005:
        return 'RANGE'

    else:
        return 'WEAK'

df_15m['REGIME'] = df_15m.apply(market_regime, axis=1)

# ======================
# 对齐到 1m
# ======================
df_1m['ALLOW_LONG'] = df_15m['ALLOW_LONG'].reindex(df_1m.index, method='ffill')
df_1m['ALLOW_SHORT'] = df_15m['ALLOW_SHORT'].reindex(df_1m.index, method='ffill')
df_1m['REGIME'] = df_15m['REGIME'].reindex(df_1m.index, method='ffill')

# ======================
# 1m 技术因子
# ======================
df_1m['EMA20'] = EMA(df_1m['close'], 20)
df_1m['ATR'] = ATR(df_1m)

df_1m['VOL_SPIKE'] = (
    df_1m['volume'] >
    1.15 * df_1m['volume'].rolling(20).mean()
)

# ======================
# 回测主循环
# ======================
trades = []
position = None

for t in range(50, len(df_1m)):
    row = df_1m.iloc[t]

    # ===== 开仓 =====
    if position is None:

        # 强多市场只做多
        if (
            row['REGIME'] == 'STRONG_UP' and
            row['ALLOW_LONG'] and
            row['low'] <= row['EMA20'] and
            row['VOL_SPIKE']
        ):
            entry = row['close']
            stop = entry - ATR_MULTIPLIER * row['ATR']
            position = {
                'side': 'long',
                'entry_price': entry,
                'stop': stop,
                'entry_time': df_1m.index[t]
            }

        # 强空市场只做空
        elif (
            row['REGIME'] == 'STRONG_DOWN' and
            row['ALLOW_SHORT'] and
            row['high'] >= row['EMA20'] and
            row['VOL_SPIKE']
        ):
            entry = row['close']
            stop = entry + ATR_MULTIPLIER * row['ATR']
            position = {
                'side': 'short',
                'entry_price': entry,
                'stop': stop,
                'entry_time': df_1m.index[t]
            }

    # ===== 持仓管理 =====
    else:
        low, high = row['low'], row['high']

        if position['side'] == 'long':
            recent_lows = df_1m['low'].iloc[t-HL_LOOKBACK:t]
            hl_stop = recent_lows.min() * HL_BUFFER
            position['stop'] = max(position['stop'], hl_stop)

            if low <= position['stop']:
                exit_price = position['stop']
                ret = (exit_price - position['entry_price']) / position['entry_price']
                net = LEVERAGE * ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'long',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net
                })
                position = None

        else:
            recent_highs = df_1m['high'].iloc[t-HL_LOOKBACK:t]
            hh_stop = recent_highs.max() * HH_BUFFER
            position['stop'] = min(position['stop'], hh_stop)

            if high >= position['stop']:
                exit_price = position['stop']
                ret = (position['entry_price'] - exit_price) / position['entry_price']
                net = LEVERAGE * ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'short',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net
                })
                position = None

# ======================
# 结果统计
# ======================
df_trades = pd.DataFrame(trades)
# print("15分钟数据预览:")
# print(df_15m.head())

# print("1分钟数据预览：")
# print(df_1m.head())

print("交易次数:", len(df_trades))
print("胜率:", (df_trades['return'] > 0).mean())
print("累计收益:", df_trades['return'].sum())

equity = df_trades['return'].cumsum()
drawdown = equity.cummax() - equity
print("最大回撤:", drawdown.max())

print("\n按方向统计：")
print(df_trades.groupby('side')['return'].agg(['count', 'mean', 'sum']))


/var/folders/h_/2rpkgm1n1y5gpfss71sfwmgr0000gn/T/ipykernel_58993/665584864.py:48: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_15m = df_1m.resample('15T').agg({


交易次数: 217
胜率: 0.3686635944700461
累计收益: 7.805790198576991
最大回撤: 1.7917867754147876

按方向统计：
       count      mean       sum
side                            
long      97  0.036056  3.497397
short    120  0.035903  4.308393
